In [0]:
gold_df = spark.table("gold_interaction_matrix")

display(gold_df.limit(5))

In [0]:
bronze_products = spark.table("bronze_products")

In [0]:
silver_df = spark.table("silver_customer_products")

In [0]:
from pyspark.sql.functions import count, log1p

gold_df = (
    silver_df
    .groupBy("user_id","product_id")
    .agg(count("*").alias("purchase_count"))
    .withColumn("rating", log1p("purchase_count"))
)

In [0]:
train_data, test_data = gold_df.randomSplit([0.8, 0.2], seed=42)

print(train_data.count())
print(test_data.count())

In [0]:
from pyspark.ml.recommendation import ALS
als = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",

    implicitPrefs=True,

    rank=30,
    maxIter=15,
    regParam=0.05,

    coldStartStrategy="drop"
)

In [0]:
als_model = als.fit(train_data)

In [0]:
predictions = als_model.transform(test_data)

display(predictions.limit(10))

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)

print("RMSE =", rmse)

MODEL-1

In [0]:
als2 = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",

    implicitPrefs=False,

    rank=30,
    maxIter=15,
    regParam=0.05,

    coldStartStrategy="drop"
)

In [0]:
alsmodel = als2.fit(train_data)

In [0]:
predictions2 = alsmodel.transform(test_data)

display(predictions2.limit(10))

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions2)

print("RMSE =", rmse)

## MODEL-2

In [0]:
als3 = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",

    implicitPrefs=False,

    rank = 50,
    maxIter = 20,
    regParam = 0.01,

    coldStartStrategy="drop"
)

In [0]:
als__model = als3.fit(train_data)

In [0]:
predictions3 = als__model.transform(test_data)

display(predictions3.limit(10))

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions3)

print("RMSE =", rmse)

## Generate Recommendations

In [0]:
als__model.recommendForAllUsers(5)

In [0]:
print(type(als__model))

In [0]:
als__model.userFactors.show(5)

als__model.itemFactors.show(5)

In [0]:
gold_df.select("user_id").distinct().show(20)

In [0]:
user_id= 53003

In [0]:
purchased = (

    silver_df

    .filter(silver_df.user_id==user_id)

    .select("product_id","product_name")
    .distinct()

)

display(purchased)

In [0]:
products_df = spark.table("bronze_products")

products_df.show(5)

In [0]:
candidate_products = products_df.select("product_id")

candidate_products.show(5)

In [0]:
from pyspark.sql.functions import lit

user_products = (
    candidate_products
    .withColumn("user_id", lit(user_id))
)

display(user_products.limit(5))

In [0]:
pred = als__model.transform(user_products)

display(pred.limit(10))

In [0]:
recommended = (
    pred.join(
        purchased.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

display(recommended.limit(5))

In [0]:
top5 = (
    recommended
    .orderBy(
        recommended.prediction.desc()
    )
    .limit(5)
)

display(top5)

In [0]:
final_rec = (
    top5
    .join(products_df, on="product_id")
)

display(final_rec)

In [0]:
als__model.save(
"/Volumes/workspace/default/e-commerce_recommendation/best_als_model"
)

In [0]:
top5.printSchema()

In [0]:
top5_named = (
    top5.join(
        bronze_products,
        on="product_id",
        how="left"
    )
)

display(top5_named)

In [0]:
top5_named.write.mode("overwrite").saveAsTable("recommended_products")

In [0]:
top5_named = (
    top5.join(
        bronze_products,
        on="product_id",
        how="left"
    )
)

display(top5_named)

In [0]:
spark.sql("SHOW TABLES").filter("tableName='recommended_products'").show()

In [0]:
pred.show()

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

mae_evaluator = RegressionEvaluator(
    metricName="mae",
    labelCol="rating",
    predictionCol="prediction"
)

mae = mae_evaluator.evaluate(predictions3)

print("MAE =", mae)

In [0]:
# Generate Top-10 recommendations for every user
top10_recommendations = als_model.recommendForAllUsers(10)

display(top10_recommendations.limit(5))

In [0]:
predictions3.printSchema()